# Colab Run & Implementation Template

This notebook contains cells to: clone the repository, install Colab-specific dependencies, set ngrok and service tokens, run the Colab summary service, and basic project scaffolding examples.


## 0) CS6493 GCP Cost & Region Reminder

> Follow this checklist before starting runtime to avoid burning credits.

- Use template: `template-g2-standard-16-L4`
- Region must be: `asia-east1`
- Do not create/use runtimes in `us-east1` or `us-central1` (especially V100-related configs are expensive).
- Delete unnecessary templates/runtimes in other regions.
- Manually stop runtime after each session, even if idle auto-shutdown exists.

If your current runtime is not in `asia-east1`, stop it and recreate with the required template first.

## 1) Environment setup (Colab + branch)

Run the next Python code cell to clone and switch to the current branch `sunbaozheng`.

Tip: if `git checkout sunbaozheng` fails, run `!git branch -a` to verify branch visibility and `!git fetch origin sunbaozheng` first.

In [ ]:
# Execute setup: clone repo, switch branch, install dependencies
from pathlib import Path

REPO_URL = 'https://github.com/Yks-ydl/smart_meeting_assistant.git'
REPO_DIR = Path('/content/smart_meeting_assistant')
TARGET_BRANCH = 'sunbaozheng'

# Keep setup idempotent: reuse existing clone when rerunning the cell.
if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'Reusing existing repository: {REPO_DIR}')

%cd /content/smart_meeting_assistant
!git fetch --all
!git checkout {TARGET_BRANCH}
!pip install -r scripts/colab/requirements-colab.txt


## 2) Set ngrok token and generate secure service token

Hardcoding secrets is allowed in this notebook for quick experiments.

Set one application token (hardcoded or generated) and share it as:
- Colab runtime env: `COLAB_SUMMARY_AUTH_TOKEN`
- local gateway env: `SUMMARY_REMOTE_AUTH_TOKEN`

This step reuses an existing ngrok tunnel when possible to avoid duplicate endpoint conflicts (`ERR_NGROK_334`).

In [4]:
# Execute auth setup: set ngrok token, set summary auth token, and prepare/reuse tunnel
import os
from pyngrok import ngrok

# Hardcoded tokens are allowed in this notebook.
# Replace these literals with real values before running in Colab.
ngrok_token = '3CFZtli2Xu1cQthT0oUNtkn8aHC_2yvGt8bGxH1x144CJFLth'
summary_auth_token = 'T-KKGow3l_UaUaCCOWN76ljSok0u1Lcp5KJf8ZqWyDM'

os.environ['NGROK_AUTHTOKEN'] = ngrok_token.strip()
os.environ['COLAB_SUMMARY_AUTH_TOKEN'] = summary_auth_token.strip()

print('Using COLAB_SUMMARY_AUTH_TOKEN (copy to local .env as SUMMARY_REMOTE_AUTH_TOKEN):')
print(os.environ['COLAB_SUMMARY_AUTH_TOKEN'])

ngrok.set_auth_token(os.environ['NGROK_AUTHTOKEN'])
service_port = int(os.environ.get('COLAB_SUMMARY_PORT', '8002'))

existing_tunnels = ngrok.get_tunnels()
if existing_tunnels:
    public_url = existing_tunnels[0].public_url
    print('Reusing existing ngrok tunnel:', public_url)
else:
    public_url = ngrok.connect(service_port, 'http').public_url
    print(f'Created new ngrok tunnel (port {service_port}):', public_url)

os.environ['COLAB_PUBLIC_URL'] = public_url
print('COLAB_PUBLIC_URL exported for the service startup cell.')

Using COLAB_SUMMARY_AUTH_TOKEN (copy to local .env as SUMMARY_REMOTE_AUTH_TOKEN):
T-KKGow3l_UaUaCCOWN76ljSok0u1Lcp5KJf8ZqWyDM
Created new ngrok tunnel (port 8002): https://rekindle-habitant-disparate.ngrok-free.dev
COLAB_PUBLIC_URL exported for the service startup cell.


## 3) Start the FastAPI Colab service

Run the service entrypoint which loads the model and starts `uvicorn`. Recommended invocation is module mode to preserve package import context.

If `COLAB_PUBLIC_URL` is already set by Step 2, the entrypoint reuses that tunnel and avoids creating a second endpoint.

If you still see `ERR_NGROK_334`, rerun Step 2 first to refresh/reuse the existing tunnel URL, then rerun this step.

In [ ]:
# Execute service startup (foreground)
import os

public_url = os.environ.get('COLAB_PUBLIC_URL', '').strip()
if public_url:
    os.environ['COLAB_PUBLIC_URL'] = public_url
    print('Using COLAB_PUBLIC_URL from environment:', public_url)
else:
    print('COLAB_PUBLIC_URL is not set; entrypoint will create or reuse tunnel.')

!python -u -m scripts.colab.colab_entry

Using COLAB_PUBLIC_URL from environment: https://rekindle-habitant-disparate.ngrok-free.dev
[ColabEntry] creating app and loading model...
config.json: 1.69kB [00:00, 3.62MB/s]
tokenizer_config.json: 100% 479/479 [00:00<00:00, 1.21MB/s]
vocab.txt: 259kB [00:00, 16.2MB/s]
special_tokens_map.json: 100% 156/156 [00:00<00:00, 627kB/s]
model.safetensors: 100% 561M/561M [00:06<00:00, 90.0MB/s]
Loading weights: 100% 260/260 [00:00<00:00, 1017.70it/s, Materializing param=model.shared.weight]                                 
t=2026-04-13T03:13:21+0000 lvl=warn msg="can't bind default web address, trying alternatives" obj=web addr=127.0.0.1:4040
t=2026-04-13T03:13:21+0000 lvl=warn msg="failed to start tunnel" pg=/api/tunnels id=34947d0d399f50e8 err="failed to start tunnel: The endpoint 'https://rekindle-habitant-disparate.ngrok-free.dev' is already online. Either\n1. stop your existing endpoint first, or\n2. start both endpoints with `--pooling-enabled` to load balance between them.\r\n\r\nERR_N

## 4) Project structure and minimal boilerplate

Create minimal folders and files used by the service. Run the next code cell to generate scaffolding files.

In [ ]:
from pathlib import Path

proj = Path('.')
src = proj / 'src'
tests = proj / 'tests'
src.mkdir(exist_ok=True)
tests.mkdir(exist_ok=True)

# Minimal package init
pkg = src / 'summary_service'
pkg.mkdir(exist_ok=True)
(pkg / '__init__.py').write_text('# summary_service package')

# Write a tiny README
Path('README_COLAB.md').write_text('# Colab run notes\nUse this file for quick notes')
print('Created minimal structure under', proj)


## 5) Implement a tiny core module (example)

Write a small summarizer wrapper module that can be used for local testing. In real usage, replace implementation with the transformer model loader in `colab_model_runtime.py`.

In [ ]:
from pathlib import Path

core_code = '''"""Minimal summarizer wrapper for quick tests"""

def summarize_text(text: str) -> str:
    """Return a very short placeholder summary for testing."""
    if not text or not text.strip():
        return ''
    # naive placeholder: return first sentence
    return text.strip().split('.')[:1][0]
'''

Path('src/summary_service/core.py').write_text(core_code)
print('Wrote src/summary_service/core.py')

## 6) Command-line entry point example

Create a simple CLI wrapper to call the `summarize_text` function.

In [ ]:
from pathlib import Path

cli_code = '''import argparse
from summary_service.core import summarize_text

if __name__ == '__main__':
    p = argparse.ArgumentParser()
    p.add_argument('--text', required=True)
    args = p.parse_args()
    print(summarize_text(args.text))
'''
Path('src/summary_service/cli.py').write_text(cli_code)
print('Wrote CLI at src/summary_service/cli.py')


## 7) Unit tests (pytest)

Create simple pytest tests for the core summarizer.

In [ ]:
from pathlib import Path

test_code = '''from summary_service.core import summarize_text


def test_empty():
    assert summarize_text('') == ''


def test_basic():
    assert summarize_text('Hello world. More text.') == 'Hello world'
'''
Path('tests/test_core.py').write_text(test_code)
print('Wrote tests/test_core.py')


## 8) Run tests and view output

Run tests from the notebook or shell. In Colab, use the next code cell to execute pytest commands.

In [ ]:
# Run full pytest suite
!pytest -q

# Run single test file
!pytest -q tests/test_core.py


## 9) Logging and error handling

Add basic logging configuration to the core module and demonstrate raising/catching an error.

In [ ]:
from pathlib import Path

logging_code = '''import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def example():
    logger.info('Example log')
    raise ValueError('Demonstration error')
'''
Path('src/summary_service/logging_demo.py').write_text(logging_code)
print('Wrote logging demo')


## 10) Continuous integration (GitHub Actions)

Write a minimal workflow that installs Python and runs tests.

In [ ]:
from pathlib import Path

workflow = '''name: Python CI
on: [push]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Set up Python
        uses: actions/setup-python@v4
        with:
          python-version: '3.10'
      - name: Install dependencies
        run: |
          pip install -r scripts/colab/requirements-colab.txt
      - name: Run tests
        run: pytest -q
'''
Path('.github/workflows/python-ci.yml').parent.mkdir(parents=True, exist_ok=True)
Path('.github/workflows/python-ci.yml').write_text(workflow)
print('Wrote .github/workflows/python-ci.yml')


## 11) Packaging and distribution (build)

Create minimal `pyproject.toml` and show commands to build a wheel and sdist.

In [ ]:
from pathlib import Path

pyproject = '''[build-system]
requires = ["setuptools>=61.0", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "smart_meeting_summary_test"
version = "0.0.1"
readme = "README_COLAB.md"
'''
Path('pyproject.toml').write_text(pyproject)
print('Wrote pyproject.toml')

# Commands to build (uncomment to run)
# !python -m pip install build
# !python -m build


## 12) Notebook demo: import and run

Quick demo that imports the local package and runs the summarizer function.

In [ ]:
import sys

sys.path.insert(0, 'src')
from summary_service.core import summarize_text

print(summarize_text('This is a demo. It should return the first sentence.'))